# Giai đoạn 4 — Chia dữ liệu thời gian và cạnh âm

Notebook dùng dữ liệu Giai đoạn 3 (`interactions_filtered.csv`, `F.npy`) để tạo các tập sự kiện theo thời gian, hai chế độ đánh giá và cạnh âm `random`/`historical`. `F.npy` là đặc trưng cạnh TF-IDF/SVD 172 chiều, **không phải LIWC gốc**. Mọi tương tác được giữ lại; không gộp cặp lặp. Sự kiện được sắp xếp ổn định theo `created_utc`, sau đó chia theo vị trí 70%/85% số sự kiện. Nếu nhiều sự kiện có cùng timestamp, thứ tự dòng ban đầu là tie-breaker; `event_id` cho phép phân biệt chúng.

Negative được lấy chỉ từ lịch sử **trước** sự kiện mục tiêu. `random` là subreddit người dùng chưa từng tham gia; `historical` là subreddit đã tham gia nhưng không phải subreddit đích ở sự kiện này. Nếu một loại không có ứng viên thì không tạo dòng negative cho sự kiện đó; notebook báo số lượng thiếu, không thay bằng loại khác.

Trên Kaggle, bật Internet để cell đầu clone repository công khai từ GitHub. Attach dataset Stage 3 như Kaggle Input (trừ khi đã đưa `interactions_filtered.csv` và `F.npy` vào repo). Cell đầu tiên thực hiện bootstrap, dò dữ liệu; các cell tiếp theo tạo split, train và đánh giá. Các file phân chia/đánh giá trong `OUTPUT_DIR` là chuẩn cho Giai đoạn 4. Runner TIGER tự chia theo quantile và tự lấy negative, nên metric runner **không** phải metric `Test-new`/`Test-all` của notebook.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd

# Kaggle Notebook settings: enable Internet; attach the Stage 3 data as an Input.
REPO_URL = 'https://github.com/HongHa-1606/www2023tiger.git'
REPO_BRANCH = 'main'  # Change if the notebook is on a different branch.
KAGGLE_WORKING = Path('/kaggle/working')
if KAGGLE_WORKING.exists():
    REPO_DIR = KAGGLE_WORKING / 'www2023tiger'
    if not (REPO_DIR / 'train_self_supervised.py').is_file():
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            raise FileExistsError(f'{REPO_DIR} exists but is not a TIGER checkout; remove/rename that folder or choose another REPO_DIR.')
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path.cwd().resolve()
if not (REPO_DIR / 'train_self_supervised.py').is_file():
    raise FileNotFoundError(f'TIGER repository checkout is missing train_self_supervised.py: {REPO_DIR}')

# Set DATA_DIR explicitly if the attached Kaggle dataset uses a custom path/name.
DATA_DIR = None  # Optionally set to an explicit folder containing interactions_filtered.csv and F.npy.
OUTPUT_DIR = Path('/kaggle/working/stage4_reddit') if Path('/kaggle/working').exists() else Path.cwd() / 'stage4_reddit'
NEGATIVE_SEED = 2026

def find_data_dir():
    if DATA_DIR is not None:
        candidate = Path(DATA_DIR).expanduser()
        if not (candidate / 'interactions_filtered.csv').is_file() or not (candidate / 'F.npy').is_file():
            raise FileNotFoundError(f'DATA_DIR must contain interactions_filtered.csv and F.npy: {candidate}')
        return candidate

    candidates = [
        REPO_DIR / 'reddit_datatfidf - Copy',
        REPO_DIR / 'data',
        REPO_DIR,
        Path.cwd() / 'reddit_datatfidf - Copy',
        Path.cwd(),
        Path('/kaggle/input/reddit-datatfidf-copy'),
    ]
    for base in (Path('/kaggle/input'), Path.cwd()):
        if base.exists():
            for pattern in ('*/interactions_filtered.csv', '*/*/interactions_filtered.csv'):
                candidates.extend(p.parent for p in base.glob(pattern))
    for candidate in candidates:
        if (candidate / 'interactions_filtered.csv').is_file() and (candidate / 'F.npy').is_file():
            return candidate
    raise FileNotFoundError('Không tìm thấy interactions_filtered.csv và F.npy. Hãy attach dataset Stage 3 trong Kaggle (Add Input) hoặc đặt DATA_DIR ở cell cấu hình.')

DATA_DIR = find_data_dir()
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'REPO_DIR: {REPO_DIR}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR.resolve()}')

In [ ]:
# Read all interactions and align each feature row with the original CSV row before sorting.
inter = pd.read_csv(DATA_DIR / 'interactions_filtered.csv')
required = {'author', 'subreddit', 'created_utc'}
missing = required.difference(inter.columns)
if missing:
    raise ValueError(f'Các cột bắt buộc bị thiếu: {sorted(missing)}')
if inter.empty:
    raise ValueError('interactions_filtered.csv không có sự kiện nào.')
if inter[['author', 'subreddit', 'created_utc']].isna().any().any():
    raise ValueError('author, subreddit, created_utc không được chứa giá trị rỗng.')

features = np.load(DATA_DIR / 'F.npy', allow_pickle=False)
if features.ndim != 2 or features.shape != (len(inter), 172):
    raise ValueError(f'F.npy phải có shape ({len(inter)}, 172); nhận được {features.shape}.')
if not np.isfinite(features).all():
    raise ValueError('F.npy chứa NaN hoặc infinity.')

inter = inter.copy()
inter['_source_row'] = np.arange(len(inter), dtype=np.int64)
inter['created_utc'] = pd.to_numeric(inter['created_utc'], errors='raise')
inter = inter.sort_values(['created_utc', '_source_row'], kind='mergesort').reset_index(drop=True)
feature_rows = features[inter['_source_row'].to_numpy(dtype=np.int64)]

# User and subreddit IDs occupy disjoint, positive ranges; 0 remains TIGER's padding ID.
authors = sorted(inter['author'].astype(str).unique())
subreddits = sorted(inter['subreddit'].astype(str).unique())
author_to_id = {name: idx + 1 for idx, name in enumerate(authors)}
subreddit_offset = len(authors) + 1
subreddit_to_id = {name: subreddit_offset + idx for idx, name in enumerate(subreddits)}
node_map_df = pd.DataFrame(
    [{'node_id': node_id, 'node_type': 'author', 'name': name} for name, node_id in author_to_id.items()]
    + [{'node_id': node_id, 'node_type': 'subreddit', 'name': name} for name, node_id in subreddit_to_id.items()]
)
node_map_df.to_csv(OUTPUT_DIR / 'stage4_node_map.csv', index=False)
inter['author'] = inter['author'].astype(str)
inter['subreddit'] = inter['subreddit'].astype(str)
inter['author_id'] = inter['author'].map(author_to_id).astype(np.int64)
inter['subreddit_id'] = inter['subreddit'].map(subreddit_to_id).astype(np.int64)
inter['event_id'] = np.arange(len(inter), dtype=np.int64)
inter['edge_feature_idx'] = inter['event_id']

assert len(inter) == len(feature_rows)
assert inter['created_utc'].is_monotonic_increasing
assert inter['event_id'].is_unique
print(f"Sự kiện: {len(inter):,} | users: {len(authors):,} | subreddits: {len(subreddits):,} | F: {feature_rows.shape}")
print(f"Timestamp: {inter['created_utc'].iloc[0]} → {inter['created_utc'].iloc[-1]}")

In [ ]:
# Positional cutoffs produce 70/15/15 by event count while retaining chronological order.
n_events = len(inter)
train_end = int(np.floor(0.70 * n_events))
val_end = int(np.floor(0.85 * n_events))
split_labels = np.full(n_events, 'test', dtype=object)
split_labels[:train_end] = 'train'
split_labels[train_end:val_end] = 'val'
inter['split'] = split_labels

event_columns = ['event_id', 'author', 'subreddit', 'author_id', 'subreddit_id', 'created_utc', 'edge_feature_idx', 'split']
for split_name in ('train', 'val', 'test'):
    part = inter.loc[inter['split'].eq(split_name), event_columns].copy()
    part.to_csv(OUTPUT_DIR / f'{split_name}_events.csv', index=False)
    np.save(OUTPUT_DIR / f'F_{split_name}.npy', feature_rows[part['event_id'].to_numpy(dtype=np.int64)])

split_summary = inter.groupby('split', sort=False).agg(
    n_events=('event_id', 'size'), first_t=('created_utc', 'min'), last_t=('created_utc', 'max')
).reindex(['train', 'val', 'test'])
print(split_summary.to_string())
print(f'Cut positions: train_end={train_end}, val_end={val_end}; all {n_events:,} events retained.')

In [ ]:
# Build evaluation sets and time-aware negatives in one chronological pass.
rng = np.random.default_rng(NEGATIVE_SEED)
all_subreddit_ids = np.asarray(sorted(subreddit_to_id.values()), dtype=np.int64)
history_by_author = {}
seen_pairs = set()
eval_rows = {(s, mode): [] for s in ('val', 'test') for mode in ('all', 'new')}
negative_rows = {(s, mode, kind): [] for s in ('val', 'test') for mode in ('all', 'new') for kind in ('random', 'historical')}
negative_missing = {(s, mode, kind): 0 for s in ('val', 'test') for mode in ('all', 'new') for kind in ('random', 'historical')}

for row in inter.itertuples(index=False):
    user_id = int(row.author_id)
    positive_id = int(row.subreddit_id)
    pair = (user_id, positive_id)
    prior_history = history_by_author.get(user_id, set())
    is_new_pair = pair not in seen_pairs
    split_name = row.split

    if split_name in ('val', 'test'):
        event = {
            'event_id': int(row.event_id), 'author': row.author, 'subreddit': row.subreddit,
            'author_id': user_id, 'subreddit_id': positive_id,
            'created_utc': row.created_utc, 't': row.created_utc,
            'edge_feature_idx': int(row.edge_feature_idx), 'split': split_name,
        }
        eval_rows[(split_name, 'all')].append(event)
        if is_new_pair:
            eval_rows[(split_name, 'new')].append(event)

        random_candidates = all_subreddit_ids[~np.isin(all_subreddit_ids, list(prior_history | {positive_id}))]
        historical_candidates = np.asarray(sorted(prior_history - {positive_id}), dtype=np.int64)
        sampled = {}
        for kind, candidates in (('random', random_candidates), ('historical', historical_candidates)):
            sampled[kind] = int(rng.choice(candidates)) if len(candidates) else None

        for mode in ('all', 'new'):
            if mode == 'new' and not is_new_pair:
                continue
            for kind, negative_id in sampled.items():
                if negative_id is None:
                    negative_missing[(split_name, mode, kind)] += 1
                    continue
                negative_rows[(split_name, mode, kind)].append({
                    'event_id': int(row.event_id), 'author_id': user_id,
                    'positive_subreddit_id': positive_id,
                    'negative_subreddit_id': negative_id,
                    'negative_subreddit': subreddits[negative_id - subreddit_offset],
                    'created_utc': row.created_utc, 't': row.created_utc,
                })

    # Update only after candidates are selected: the current positive is never a historical negative.
    history_by_author.setdefault(user_id, set()).add(positive_id)
    seen_pairs.add(pair)

for split_name in ('val', 'test'):
    for mode in ('all', 'new'):
        eval_df = pd.DataFrame(eval_rows[(split_name, mode)], columns=event_columns[:-1] + ['t', 'split'])
        eval_df.to_csv(OUTPUT_DIR / f'{split_name}_{mode}.csv', index=False)
        for kind in ('random', 'historical'):
            neg_df = pd.DataFrame(negative_rows[(split_name, mode, kind)], columns=[
                'event_id', 'author_id', 'positive_subreddit_id', 'negative_subreddit_id',
                'negative_subreddit', 'created_utc', 't',
            ])
            neg_df.to_csv(OUTPUT_DIR / f'{split_name}_{mode}_negative_{kind}.csv', index=False)

summary_rows = []
for key, events in eval_rows.items():
    split_name, mode = key
    summary_rows.append({
        'split': split_name, 'evaluation': f'{split_name}-{mode}', 'events': len(events),
        'negative_random': len(negative_rows[(split_name, mode, 'random')]),
        'random_unavailable': negative_missing[(split_name, mode, 'random')],
        'negative_historical': len(negative_rows[(split_name, mode, 'historical')]),
        'historical_unavailable': negative_missing[(split_name, mode, 'historical')],
    })
evaluation_summary = pd.DataFrame(summary_rows)
print(evaluation_summary.to_string(index=False))
print(f'Đã ghi các bảng event/negative tại: {OUTPUT_DIR.resolve()}')

## Chuẩn bị file tương thích TIGER và chạy train tùy chọn

Cell tiếp theo dò source code TIGER trong working directory hoặc `/kaggle/input`. Nếu repository được attach read-only, notebook tạo bản làm việc trong `/kaggle/working/www2023tiger`. File `ml_reddit.csv` giữ mọi event và `ml_reddit.npy` có hàng zero ở index 0 theo quy ước feature ID của TIGER. Node features đặt zero 172 chiều để khớp runner hiện có; các node features GCN ở Giai đoạn 3 được giữ trong đầu ra giai đoạn đó, không tự chuyển đổi ở đây.

Đặt `RUN_TIGER_TRAINING = True` ở cell cuối để chạy huấn luyện. TIGER runner hiện tại **không đọc** các CSV `Test-new`/`Test-all` hoặc negative của notebook làm bộ đánh giá; hãy dùng các file `*_new.csv`, `*_all.csv`, `*_negative_*.csv` cho baseline truy vấn trạng thái tại `t`.

In [ ]:
def find_repo_root():
    candidates = [Path.cwd(), Path('/kaggle/input/www2023tiger')]
    for base in (Path('/kaggle/input'), Path.cwd()):
        if base.exists():
            for pattern in ('*/train_self_supervised.py', '*/*/train_self_supervised.py'):
                candidates.extend(p.parent for p in base.glob(pattern))
    for candidate in candidates:
        if (candidate / 'train_self_supervised.py').is_file():
            return candidate.resolve()
    raise FileNotFoundError('Không tìm thấy train_self_supervised.py. Hãy attach repository www2023tiger trong Kaggle hoặc chạy notebook từ thư mục repository.')

source_repo = find_repo_root()
try:
    source_repo.relative_to(Path('/kaggle/input').resolve())
    source_is_read_only = True
except ValueError:
    source_is_read_only = False
working_repo = Path('/kaggle/working/www2023tiger') if source_is_read_only else source_repo
if source_repo != working_repo:
    if not (working_repo / 'train_self_supervised.py').is_file():
        shutil.copytree(
            source_repo, working_repo, dirs_exist_ok=True,
            ignore=shutil.ignore_patterns('.git', '__pycache__', 'saved_models', 'saved_checkpoints', 'results'),
        )
PROJECT_ROOT = working_repo.resolve()
if not (PROJECT_ROOT / 'train_self_supervised.py').is_file():
    raise FileNotFoundError(f'TIGER source is not available in writable project folder: {PROJECT_ROOT}')

tiger_data_dir = PROJECT_ROOT / 'data'
tiger_data_dir.mkdir(parents=True, exist_ok=True)
tiger_inter = pd.DataFrame({
    'u': inter['author_id'].to_numpy(dtype=np.int64),
    'i': inter['subreddit_id'].to_numpy(dtype=np.int64),
    'ts': inter['created_utc'].to_numpy(dtype=np.float64),
    'label': np.zeros(len(inter), dtype=np.int64),
    'idx': inter['event_id'].to_numpy(dtype=np.int64) + 1,
})
tiger_inter.to_csv(tiger_data_dir / 'ml_reddit.csv', index=False)
np.save(tiger_data_dir / 'ml_reddit.npy', np.vstack((np.zeros((1, 172), dtype=np.float32), feature_rows.astype(np.float32))))
n_nodes = len(authors) + len(subreddits) + 1
np.save(tiger_data_dir / 'ml_reddit_node.npy', np.zeros((n_nodes, 172), dtype=np.float32))

assert len(tiger_inter) == len(inter)
assert tiger_inter['idx'].iloc[0] == 1 and tiger_inter['idx'].iloc[-1] == len(inter)
assert np.load(tiger_data_dir / 'ml_reddit.npy', mmap_mode='r').shape == (len(inter) + 1, 172)
print(f'TIGER inputs ready: {tiger_data_dir}')
print(f'Events: {len(tiger_inter):,}; nodes incl. padding: {n_nodes:,}; edge features: 172')

In [ ]:
# Training runs when this cell is executed; set False to prepare data only.
RUN_TIGER_TRAINING = True
N_EPOCHS = 5
BATCH_SIZE = 200

if RUN_TIGER_TRAINING:
    import torch
    gpu = '0' if torch.cuda.is_available() else '-1'
    command = [
        sys.executable, 'train_self_supervised.py',
        '--data', 'reddit', '--root', str(PROJECT_ROOT), '--gpu', gpu,
        '--n_epochs', str(N_EPOCHS), '--bs', str(BATCH_SIZE),
        '--restart_prob', '0', '--restarter_type', 'static',
    ]
    print(f"Training on {'GPU ' + gpu if gpu != '-1' else 'CPU'}; outputs are written under {PROJECT_ROOT}.")
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('Training skipped. Set RUN_TIGER_TRAINING = True and rerun this cell to start TIGER.')

## Giai đoạn 6 — Đánh giá online sau huấn luyện

Cell tiếp theo nạp checkpoint tốt nhất, phát lại train để khôi phục trạng thái, rồi duyệt validation theo thời gian để chọn `tau` (Popularity suy giảm) và `W` (EdgeBank time-window). Khi đánh giá test, mỗi sự kiện được chấm điểm trước, cập nhật trạng thái sau. MRR và Hits@1/5/10 xếp hạng trên toàn bộ ứng viên của loại negative tương ứng; AUC/AP dùng một negative lấy mẫu trên mỗi positive. Negative random được đánh giá với 5 seed; TIGER và các baseline lịch sử chạy một lượt xác định.

Bảng kết quả tách Test-new/Test-all × random/historical, so sánh TIGER, Popularity không suy giảm/suy giảm và EdgeBank unlimited/time-window. CSV kết quả và biểu đồ MRR theo tuần được lưu vào `OUTPUT_DIR`. Replay theo từng sự kiện và chấm toàn bộ ứng viên có thể chạy lâu trên Kaggle.

In [ ]:
evaluation_command = [
    sys.executable,
    str(PROJECT_ROOT / 'stage6_eval.py'),
    '--root', str(PROJECT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
]
print('Chạy đánh giá online Stage 6 sau huấn luyện...')
subprocess.run(evaluation_command, cwd=PROJECT_ROOT, check=True)